# Parkinson’s Disease Detection via Acoustic Analysis
This notebook demonstrates a Deep Learning approach to detecting Parkinson's Disease using vocal features like dysphonia via acoustic analysis.
For future reference, dysphonia refers to a voice disorder characterized by difficulty speaking, resulting in a raspy, hoarse, shaky, or strained voice, or changes in pitch and quality.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from model import ParkinsonNet
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
class ParkinsonNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ParkinsonNet, self).__init__()
        # Input is 22 (the number of voice features in the dataset)
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, output_size) # Output 1 probability (Healthy vs PD)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # We use Sigmoid at the end to get a 0.0 to 1.0 probability
        return self.fc3(x)

In [ ]:
class ParkinsonDataset(Dataset):
    def __init__(self, file_path):
        # 1. Load the data
        df = pd.read_csv(file_path)
        
        # 2. Separate features (X) and target (y)
        # 'name' is just a label, 'status' is what we want to predict
        X_raw = df.drop(columns=['name', 'status']).values
        y_raw = df['status'].values

        # 3. Scaling is CRITICAL for medical AI
        # It ensures 'Jitter' (small numbers) isn't ignored compared to 'Frequency' (large numbers)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)

        # 4. Convert to PyTorch Tensors
        self.X = torch.tensor(X_scaled, dtype=torch.float32)
        self.y = torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Training

In [ ]:
INPUT_SIZE = 22
HIDDEN_SIZE = 64
OUTPUT_SIZE = 1
LEARNING_RATE = 0.001
EPOCHS = 50
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

In [ ]:
model = ParkinsonNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
loss_history = []

print(f"Starting Training on {len(train_loader.dataset)} samples...")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels.unsqueeze(1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Avg Loss: {avg_loss:.4f}")

# Save the model weights
torch.save(model.state_dict(), "models/parkinsons_model.pth")
print("Model saved successfully!")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='Training Loss', color='blue')
plt.title('Training Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy', 'Parkinsons'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Parkinson's Prediction - Confusion Matrix")
plt.show()